In [12]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data

In [13]:
df = pd.read_csv("../dataset//Crop_recommendation.csv")
df = df.drop(columns=[c for c in df.columns if c.lower().startswith('unnamed')])
df.columns = [c.lower() for c in df.columns]

In [14]:
X = df.drop('label', axis=1)
y = df['label']
print(X.dtypes)

n                int64
p                int64
k                int64
temperature    float64
humidity       float64
ph             float64
rainfall       float64
dtype: object


# 2. Feature Scaling

In [15]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# 3. Feature Selection: Correlation Filtering

In [16]:
corr_matrix = X_scaled_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
X_corr_filtered = X_scaled_df.drop(columns=to_drop)
print("Features to drop due to high correlation:", to_drop)

Features to drop due to high correlation: []


# 4. Feature Selection: Mutual Information

In [17]:
mi_scores = mutual_info_classif(X_corr_filtered, y)
mi_df = pd.DataFrame({
    'feature': X_corr_filtered.columns,
    'mi_score': mi_scores
}).sort_values(by='mi_score', ascending=False)
print("Mutual Information Scores:\n", mi_df)

Mutual Information Scores:
        feature  mi_score
4     humidity  1.729954
2            k  1.637829
6     rainfall  1.637358
1            p  1.309545
3  temperature  1.017901
0            n  0.988657
5           ph  0.686067


# 5. Feature Selection: Random Forest Importance

In [18]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_corr_filtered, y)
importances = rf.feature_importances_
fi_df = pd.DataFrame({
    'feature': X_corr_filtered.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)
print("Tree-based Importance:\n", fi_df)

Tree-based Importance:
        feature  importance
6     rainfall    0.226885
4     humidity    0.213955
2            k    0.180255
1            p    0.150093
0            n    0.102958
3  temperature    0.073629
5           ph    0.052225


# 6. Final Feature Selection & Save Artifacts

In [23]:
all_selected = mi_df['feature'].tolist()

selected_features = [f for f in X.columns if f in all_selected]

X_selected = X_corr_filtered[selected_features]

print("Features in original order:", selected_features)


Features in original order: ['n', 'p', 'k', 'temperature', 'humidity', 'ph', 'rainfall']


In [24]:
joblib.dump(scaler, r"../models/scaler.pkl")
joblib.dump(selected_features, r"../models/selected_features.pkl")

['../models/selected_features.pkl']